# Evo-1: Proprioceptive State Utilization Analysis

## Research Question
**Hypothesis**: Proprioceptive state information is NOT being fully utilized by the integration module.

## Experimental Design
1. **Pass normal state** → Measure gradient flow to integration_module
2. **Pass zeros (ablation)** → Measure gradient flow to integration_module
3. **Compare**: If gradients barely change, state is underutilized

## What We'll Measure
- Gradient magnitude at integration_module (Evo-1's VL+state aligner)
- Gradient variance across batches
- Performance drop when state is zeroed
- Gradient sensitivity: ∂Loss/∂integration_features

**Model**: Evo-1 (0.77B parameters)  
**State Encoder**: Integration Module (cross-attention based VL+state alignment)  
**Key Metric**: If zero state → minimal gradient change = underutilization

**Why This Matters**: Evo-1 claims integration_module is critical innovation - let's verify!

---

# Part 1: Model Analysis & Hook Diagnostics

## 1. Environment Setup

In [ ]:
# Check environment
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running on Google Colab")
    !nvidia-smi
else:
    print("💻 Running locally")

In [ ]:
# Install dependencies
!pip install -q torch torchvision transformers accelerate diffusers pillow numpy matplotlib seaborn scikit-learn timm

print("✅ Dependencies installed")

In [ ]:
# Clone repository
if IN_COLAB:
    !git clone https://github.com/PrithviTM-glitch/EmbodiedLLM.git
    %cd EmbodiedLLM
    !git checkout AnalyseMultipleHooks
    %cd MultipleHooksStudy
else:
    import os
    if not os.path.exists('hooks'):
        print("⚠️ Please run from MultipleHooksStudy directory")
    else:
        print("✅ Hook files found")

## 2. Import Hook Framework

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add hooks to path
sys.path.insert(0, str(Path.cwd() / 'hooks'))

from hooks.model_specific.evo1_hooks import Evo1Hooks

print("✅ Evo-1 hook framework imported")

## 3. Load Evo-1 Model

**Note**: For analysis, we'll create a mock structure. For benchmarks, see Part 2 for full model loading.

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create mock model matching Evo-1 architecture
class MockInternVL3(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision_model = nn.Sequential(
            nn.Conv2d(3, 768, 14, stride=14),
            nn.Flatten(2),
            nn.Linear(768, 1024)
        )
        self.language_model = nn.Sequential(
            nn.Embedding(50000, 1024),
            nn.TransformerEncoderLayer(d_model=1024, nhead=8),
        )

class MockEvo1(nn.Module):
    def __init__(self):
        super().__init__()
        self.vl_backbone = MockInternVL3()
        
        # Integration Module (critical innovation)
        self.integration_module = nn.Sequential(
            nn.Linear(1024 + 7, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Linear(512, 512)
        )
        
        # Cross-Modulated Diffusion Transformer
        self.diffusion_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=512, nhead=8),
            num_layers=6
        )
        
        self.action_head = nn.Linear(512, 7)
    
    def forward(self, pixel_values, input_ids, robot_state):
        vision_features = self.vl_backbone.vision_model(pixel_values)
        lang_features = self.vl_backbone.language_model(input_ids)
        vl_features = vision_features.mean(dim=-1) + lang_features.mean(dim=1)
        
        combined = torch.cat([vl_features, robot_state], dim=-1)
        integrated = self.integration_module(combined)
        
        integrated = integrated.unsqueeze(1)
        diffusion_output = self.diffusion_transformer(integrated)
        
        actions = self.action_head(diffusion_output.squeeze(1))
        return actions

model = MockEvo1().to(device).half()
print(f"✅ Mock Evo-1 created")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 4. Discover Model Structure

In [ ]:
hook_manager = Evo1Hooks(model)
structure = hook_manager.discover_model_structure()

print("\n" + "="*60)
print("Evo-1 Model Structure Discovery")
print("="*60)
print(f"Model Name: {structure['model_name']}")
print(f"Has Proprio Encoder: {structure['has_proprio_encoder']}")
print(f"Proprio Encoder Type: {structure['proprio_encoder_type']}")

print(f"\nComponents Found:")
for component, attr in structure['components'].items():
    print(f"  - {component}: {attr}")

if structure.get('integration_module_found'):
    print("\n🎯 Integration Module FOUND!")
    print("   Key innovation in Evo-1 architecture")
print("="*60)

## 5. Attach Analysis Hooks

In [ ]:
# Attach ONLY gradient hooks (focus on integration_module)
print("Attaching gradient hooks...")

hook_manager.attach_gradient_hooks()
print("✓ Gradient flow hooks attached")
print("  → Monitoring integration_module specifically")

print("\n✅ Ready to analyze state utilization!")

## 6. Run Forward and Backward Pass

In [ ]:
# Prepare sample inputs
inputs = {
    'pixel_values': torch.randn(1, 3, 224, 224).to(device).half(),
    'input_ids': torch.randint(0, 50000, (1, 10)).to(device),
    'robot_state': torch.randn(1, 7).to(device).half()
}

model.train()
outputs = model(**inputs)
loss = outputs.mean()
loss.backward()

print("✅ Forward and backward pass completed")
print(f"Loss: {loss.item():.4f}")

## 7. Evo-1 Research Insights

In [ ]:
# Capture baseline gradient flow
results_baseline = hook_manager.get_results()
gradient_baseline = results_baseline.get('gradient_flow', {})

print("\n" + "="*60)
print("BASELINE: Gradient Flow with Normal State")
print("="*60)

# Extract integration_module gradients
baseline_integration_grad = None
if 'layer_profiles' in gradient_baseline:
    layer_profiles = gradient_baseline['layer_profiles']
    if 'integration_module' in layer_profiles:
        baseline_integration_grad = layer_profiles['integration_module']
        print("\n🎯 Integration Module Gradients (NORMAL STATE):")
        print(f"  Gradient norm: {baseline_integration_grad.get('norm', 0):.6f}")
        print(f"  Gradient mean: {baseline_integration_grad.get('mean_gradient', 0):.6f}")
        print(f"  Gradient variance: {baseline_integration_grad.get('gradient_variance', 0):.6f}")

print("="*60)

## 8. Ablation: Zero Out State Encoder Output

**Better Ablation Strategy**: Instead of zeroing the input, we zero the OUTPUT of `integration_module`. 

This directly tests: "What if the state encoder contributed nothing to the model?"

In [ ]:
# Create hook to zero out integration_module's output
ablation_handle = None

def zero_output_hook(module, input, output):
    """Replace integration_module output with zeros"""
    return torch.zeros_like(output)

# Find and hook the integration_module
for name, module in model.named_modules():
    if 'integration_module' in name:
        ablation_handle = module.register_forward_hook(zero_output_hook)
        print(f"✓ Attached ablation hook to: {name}")
        break

# Run ablation: normal input, but state encoder output is zeroed
print("\nRunning ablation (integration_module OUTPUT = zeros)...")
model.train()
hook_manager.reset()
model.zero_grad()

outputs_ablated = model(**inputs)  # Same inputs, but integration_module outputs zeros
loss_ablated = outputs_ablated.mean()
loss_ablated.backward()

# Remove ablation hook
if ablation_handle:
    ablation_handle.remove()

print(f"✓ Ablation complete - state encoder contribution removed")

# Capture ablation gradients
results_ablated = hook_manager.get_results()
gradient_ablated = results_ablated.get('gradient_flow', {})

## 9. Compare Gradients: Normal vs Ablation

In [ ]:
# Extract integration module gradients from ablation run
ablated_integration_grad = results_ablated.get('integration_module', {})
ablated_norm = ablated_integration_grad.get('gradient_norm', 0.0)

print("Integration Module Gradients (ZERO STATE)")
print(f"Gradient norm: {ablated_norm:.6f}\n")

# Calculate change
baseline_norm = baseline_integration_grad.get('gradient_norm', 0.0)
grad_change_pct = abs(baseline_norm - ablated_norm) / baseline_norm * 100 if baseline_norm > 0 else 0

print(f"{'='*60}")
print(f"COMPARISON")
print(f"{'='*60}")
print(f"Normal State Gradient:   {baseline_norm:.6f}")
print(f"Ablation Gradient:       {ablated_norm:.6f}")
print(f"Change:                  {grad_change_pct:.1f}%")
print(f"{'='*60}")

## 10. Verdict: Is Proprioceptive State Utilized?

In [ ]:
# Verdict thresholds
if grad_change_pct < 10:
    verdict = "❌ UNDERUTILIZED"
    explanation = "When integration_module output is zeroed, gradients barely change. The model doesn't rely on state encoder's contribution."
elif grad_change_pct < 30:
    verdict = "⚠️ PARTIALLY UTILIZED"
    explanation = "Some gradient sensitivity when state encoder is removed, but the dependency is weak."
else:
    verdict = "✅ WELL UTILIZED"
    explanation = "Strong gradient response when state encoder output is ablated. The model meaningfully uses state information."

print(f"\n{'='*60}")
print(f"VERDICT: {verdict}")
print(f"{'='*60}")
print(f"\nGradient Change: {grad_change_pct:.1f}%")
print(f"\n{explanation}")
print(f"\nMethodology: Measured gradient sensitivity when integration_module")
print(f"output was replaced with zeros (output ablation, not input ablation).")
print(f"{'='*60}")

# Export results
results = {
    'model': 'Evo-1 (0.77B)',
    'state_encoder': 'integration_module',
    'ablation_method': 'output_ablation',
    'baseline_grad_norm': float(baseline_norm),
    'ablated_grad_norm': float(ablated_norm),
    'gradient_change_pct': float(grad_change_pct),
    'verdict': verdict
}

import json
with open('evo1_state_utilization_results.json', 'w') as f:
    json.dump(results, f, indent=2)
    
print(f"\n✓ Results exported to: evo1_state_utilization_results.json")